# 🎮 Minecraft AI Builder - Text-to-Build

Generate Minecraft builds from text prompts like **"medieval castle with towers"**

## ✨ What This Does:

1. **Trains AI models** (~35-48 hours total)
2. **Generates text descriptions** for dataset using Gemini
3. **Learns text-to-build** mapping
4. **Generates builds** from your prompts

## 🚀 How to Use:

1. **Edit the API key** in the cell below (required)
2. **Click Runtime → Run all** and wait
3. **Download generated builds** at the end

**Training time:** ~35-48 hours on free Colab GPU

---
# ⚙️ Configuration

**IMPORTANT:** Set your Gemini API key below!

Get free key at: https://makersuite.google.com/app/apikey

In [ ]:
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"

if GEMINI_API_KEY == "YOUR_GEMINI_API_KEY_HERE":
    print("⚠️  WARNING: Replace GEMINI_API_KEY with your actual API key!")
    print("Get it here: https://makersuite.google.com/app/apikey")
else:
    print(f"✓ API Key configured: {GEMINI_API_KEY[:20]}...")

---
# 🚀 Setup (Auto-runs)

In [ ]:
!git clone https://github.com/GogaGogich123/Ai.git
%cd Ai
!git checkout capy/cap-1-bc3cdacc

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU detected! Training will be very slow.")
    print("Go to Runtime → Change runtime type → GPU")

In [ ]:
!python test_training.py

---
# 🎯 Stage 1: VQ-VAE + AI Descriptions (~9-14 hours)

This will:
- Download builds from BuildPaste
- Generate AI descriptions with Gemini
- Train VQ-VAE compression model
- Save everything to cache

In [ ]:
!python mcbuilder/train_improved_vqvae.py \
    --cache_dir ./data/cache \
    --checkpoint_dir ./checkpoints_improved \
    --chunk_size 32 \
    --overlap 4 \
    --min_blocks 800 \
    --max_blocks 50000 \
    --batch_size 4 \
    --num_workers 2 \
    --embedding_dim 128 \
    --num_embeddings 1024 \
    --num_res_blocks 3 \
    --lr 1e-4 \
    --epochs 100 \
    --save_every 10 \
    --generate_descriptions \
    --gemini_api_key $GEMINI_API_KEY \
    --description_language en

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/minecraft_ai_checkpoints
!cp -r ./checkpoints_improved /content/drive/MyDrive/minecraft_ai_checkpoints/
!cp -r ./data/cache /content/drive/MyDrive/minecraft_ai_checkpoints/

print("✓ Stage 1 checkpoint saved to Google Drive!")

---
# 🎯 Stage 2: Text-Conditioned Diffusion (~15-20 hours)

This will:
- Load pretrained CLIP text encoder
- Train diffusion with cross-attention
- Learn to generate from text prompts

In [ ]:
!python mcbuilder/train_text_to_build.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --cache_dir ./data/cache \
    --checkpoint_dir ./checkpoints_text_to_build \
    --text_encoder_type clip \
    --context_dim 512 \
    --model_channels 128 \
    --num_res_blocks 2 \
    --attention_resolutions 4 8 \
    --channel_mult 1 2 4 8 \
    --num_heads 8 \
    --dropout 0.1 \
    --timesteps 1000 \
    --chunk_size 32 \
    --batch_size 4 \
    --num_workers 2 \
    --epochs 100 \
    --learning_rate 1e-4 \
    --save_every 10

In [ ]:
!cp -r ./checkpoints_text_to_build /content/drive/MyDrive/minecraft_ai_checkpoints/

print("✓ Stage 2 checkpoint saved to Google Drive!")

---
# ✨ Generate Builds from Text!

Training complete! Now generate builds from prompts.

## Example 1: Medieval Castle

In [ ]:
!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "medieval stone castle with tall towers and fortified walls" \
    --size 64,48,64 \
    --guidance_scale 8.0 \
    --num_samples 3 \
    --validate \
    --output medieval_castle.litematic

## Example 2: Cozy Cottage

In [ ]:
!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "cozy cottage with oak planks stone fireplace and wooden furniture" \
    --size 24,20,24 \
    --guidance_scale 7.5 \
    --validate \
    --output cozy_cottage.litematic

## Example 3: Modern House

In [ ]:
!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "modern suburban house with white concrete walls and large glass windows" \
    --size 32,24,32 \
    --guidance_scale 7.5 \
    --validate \
    --output modern_house.litematic

## Example 4: Fantasy Treehouse

In [ ]:
!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "fantasy treehouse with wooden platforms bridges and leaf decorations" \
    --size 32,40,32 \
    --guidance_scale 8.0 \
    --validate \
    --output treehouse.litematic

## Example 5: Japanese Pagoda

In [ ]:
!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "traditional japanese pagoda with wooden beams and curved roofs" \
    --size 32,48,32 \
    --guidance_scale 8.5 \
    --validate \
    --output pagoda.litematic

---
## 📥 Download All Generated Builds

In [ ]:
from google.colab import files
import os

for filename in os.listdir('.'):
    if filename.endswith('.litematic'):
        print(f"Downloading {filename}...")
        files.download(filename)

print("\n✓ All builds downloaded!")

---
## 🎨 Generate Your Own Custom Build

After training completes, use this cell to generate builds from your own prompts!

In [ ]:
YOUR_PROMPT = "medieval fortress with stone walls"  # Change this!
SIZE = "32,32,32"  # X,Y,Z dimensions
GUIDANCE = 7.5  # How strictly to follow prompt (1.0-15.0)

print(f"Generating: {YOUR_PROMPT}")
print(f"Size: {SIZE}")
print(f"Guidance: {GUIDANCE}\n")

!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "{YOUR_PROMPT}" \
    --size {SIZE} \
    --guidance_scale {GUIDANCE} \
    --num_samples 3 \
    --validate \
    --output my_custom_build.litematic

---
## 💡 Prompt Ideas

Try these prompts:

**Architecture:**
- "medieval stone castle with tall towers and fortified walls"
- "modern house with glass windows and concrete structure"
- "japanese pagoda with wooden beams and curved roofs"
- "gothic cathedral with stained glass and stone arches"

**Fantasy:**
- "fantasy treehouse with wooden platforms and bridges"
- "wizard tower with magical elements and bookshelves"
- "elven palace with white marble and nature integration"

**Functional:**
- "blacksmith workshop with anvils and furnaces"
- "cozy library with bookshelves and reading area"
- "medieval tavern with wooden interior and bar"

**Tips:**
- Be specific about style and materials
- Mention key features (towers, bridges, windows)
- Use Minecraft terminology (planks, cobblestone, glass)
- Keep it 5-15 words

See [TEXT_TO_BUILD.md](https://github.com/GogaGogich123/Ai/blob/capy/cap-1-bc3cdacc/TEXT_TO_BUILD.md) and [EXAMPLES.md](https://github.com/GogaGogich123/Ai/blob/capy/cap-1-bc3cdacc/EXAMPLES.md) for more!

---
## 🎓 How to Use Builds in Minecraft

1. Install **Litematica mod** (Fabric or Forge)
2. Download `.litematic` files from cells above
3. Place files in `.minecraft/schematics/` folder
4. Open Minecraft and press **M** key
5. Load schematic and paste in world

**Enjoy your AI-generated builds! 🎮✨**